#Imports

In [0]:
notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
path_components = notebook_path.split("/")
team_folder = path_components[2]
import sys
sys.path.append(f"/Workspace/eperfectstore-prod/{team_folder}/notebooks/eperfectstore-prod/e-com/COMMON_FUNCTIONS_AND_CONSTANTS_FOLDER/")

import common_functions_and_constants as CF
import libify
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.storagelevel import StorageLevel
from datetime import datetime, timedelta
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import os
import re

# HELPERS & CONSTANTS

In [0]:
TRGT_TABLE_METRICS = 'ecom_etl.ds_samokat_availability_metrics'
TRGT_TABLE_METRCICS_CITY = 'ecom_etl.ds_samokat_availability_metrics_city'
TRGT_TABLE_ALERTS = 'ecom_etl.ds_samokat_availability_alerting'
TRGT_TABLE_FORECAST_CHECK = 'ecom_etl.ds_samokat_availability_forecast'

SPO_CPFR_BASE_URL = 'https://pepsico.sharepoint.com/teams/RussiaSPO1CustomerCollaboration/'
SPO_CPFR_BASE_DIR = 'Shared%20Documents/General/E-COM/КЛИЕНТЫ/1.САМОКАТ/CPFR/MAP для инструментов'
BSNS_MAPPING_DIR = '/dbfs/ecom/samokat/mapping'
BSNS_MAPPING_PRODUCT = BSNS_MAPPING_DIR + '/4 map ASSORT TOTAL.xlsx'
BSNS_MAPPING_WH = BSNS_MAPPING_DIR + '/MAP город СМТ- SBreg-завод.xlsx'




## MAPPING DICTS

In [0]:

WAREHOUSE_BRIDGE_CONFIG = {
    "dtype_map": {
        "GLN": str,
        "GUID": str,
        "Формат": str,
        "Customer ID": str,
        "GUID РЦ": str,
        "CUSTOMER ID22": str,
        "gln_code": str,
        "warehouse_guid": str,
        "ds_customer_id": str,
        "dc_warehouse_guid": str,
        "dc_customer_id": str,
    },
    "rename_map": {
        "GLN": "gln_code",
        "GUID": "warehouse_guid",
        "Customer ID": "ds_customer_id",
        "Формат": "warehouse_format",
        "GUID РЦ": "dc_warehouse_guid",
        "CUSTOMER ID22": "dc_customer_id",
        "gln_code": "gln_code",
        "warehouse_guid": "warehouse_guid",
        "ds_customer_id": "ds_customer_id",
        "dc_warehouse_guid": "dc_warehouse_guid",
        "dc_customer_id": "dc_customer_id",
    },
    "select_columns": [
        "gln_code",
        "ds_customer_id",
        "warehouse_guid",
        "dc_warehouse_guid",
        "dc_customer_id",
        "warehouse_format",
    ],
}

DC_EXCEPTIONS_CONFIG = {
    "dtype_map": {
        "warehouse_guid DS": str,
        "warehouse_guid DC": str,
        "warehouse_guid": str,
        "dc_warehouse_guid": str,
    },
    "rename_map": {
        "warehouse_guid DS": "warehouse_guid",
        "warehouse_guid DC": "dc_warehouse_guid",
        "warehouse_guid": "warehouse_guid",
        "dc_warehouse_guid": "dc_warehouse_guid",
    },
    "select_columns": [
        "warehouse_guid",
        "dc_warehouse_guid",
    ],
}


## HELPER FUNCS

### NAMES AND TYPES

In [0]:

def read_excel_standardized(path, sheet_name, dtype_map=None, rename_map=None, select_columns=None):
    df = pd.read_excel(
        path,
        sheet_name=sheet_name,
        dtype=dtype_map
    )
    if rename_map:
        existing_map = {old: new for old, new in rename_map.items() if old in df.columns}
        df = df.rename(columns=existing_map)

    if select_columns:
        existing_cols = [col for col in select_columns if col in df.columns]
        df = df[existing_cols]
    return df


def read_csv_standardized(path, dtype_map=None, rename_map=None, select_columns=None, **kwargs):
    df = pd.read_csv(
        path,
        dtype=dtype_map,
        **kwargs
    )
    if rename_map:
        existing_map = {old: new for old, new in rename_map.items() if old in df.columns}
        df = df.rename(columns=existing_map)
    if select_columns:
        existing_cols = [col for col in select_columns if col in df.columns]
        df = df[existing_cols]
    return df


def rename_spark_columns(df, rename_map: dict):
    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    return df


def normalize_string_id_column(df, column_name):
    if column_name in df.columns:
        df[column_name] = (
            df[column_name]
            .astype("string")
            .str.replace(" ", "", regex=False)
            .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
        )
    return df


### VOLUME ADDITION

In [0]:
def apply_volume_mapping(df, mapping_df, cols_to_scale):
    return (
        df
        .join(mapping_df, "product_guid", "left")
        .transform(
            lambda d: d.select(
                "*",
                *[
                    (F.col(c) * F.col("vol_coeff")).alias(f"{c}_vol")
                    for c in cols_to_scale
                ]
            )
        )
        .drop("vol_coeff")
    )

# RUN MODE

In [0]:
FULL_RECOMPUTE = False  # True — полный пересчёт всей истории

In [0]:
today = datetime.today()

if FULL_RECOMPUTE:
    cutoff_output = '2025-38'
    cutoff_source_date = '2025-09-15'
    write_mode = 'append'
    print("⚠️ ПОЛНЫЙ ПЕРЕСЧЁТ — все данные будут перезаписаны")
else:
    cutoff_output = (today - timedelta(weeks=9)).strftime('%G-%V')
    cutoff_source_date = (today - timedelta(weeks=13)).strftime('%Y-%m-%d')
    write_mode = 'append'
    print(f"Перезапись данных с date_week >= {cutoff_output}")
    print(f"Загрузка источников с file_date >= {cutoff_source_date}")

# CREATE FULL TABLE

## INITIAL DATA

### MAPPING

In [0]:
CF.copy_from_spo(f'{SPO_CPFR_BASE_URL}', f'{SPO_CPFR_BASE_DIR}', BSNS_MAPPING_DIR)

In [0]:
sap_vol_mapping = read_excel_standardized(
    path=BSNS_MAPPING_PRODUCT,
    sheet_name="BIG data",
    **SAP_VOL_CONFIG
)
sap_vol_mapping['vol_coeff'] = sap_vol_mapping['vol_coeff'].str.replace(',', '.').astype(float)
sap_vol_mapping = spark.createDataFrame(sap_vol_mapping).filter(F.col('is_primary') == 1).drop('is_primary')

In [0]:
products = rename_spark_columns(
    spark.table("ecom_etl.ds_samokat_products"),
    PRODUCTS_RENAME_MAP
)
new_products_pd = read_csv_standardized(
    path="MAPPING/products_upd.csv",
    sep=";",
    **NEW_PRODUCTS_CONFIG
)

new_products = spark.createDataFrame(new_products_pd)

updated_products = (
    products.alias("old")
    .join(new_products.alias("new"), "product_guid", "left")
    .select(
        F.col("old.product_guid"),
        F.col("old.product_nm"),
        F.coalesce(F.col("new.new_gtin"), F.col("old.gtin")).alias("gtin"),
        F.col("old.file_date")
    )
)

In [ ]:

products_window = Window.partitionBy('product_guid').orderBy(F.col('file_date').desc())
products_latest = (
    updated_products
    .withColumn('rn', F.row_number().over(products_window))
    .filter(F.col('rn') == 1)
    .drop('rn')
    .drop('file_date')
)

warehouses_window = Window.partitionBy('warehouse_guid').orderBy(F.col('file_date').desc())
warehouses = spark.table('ecom_etl.ds_samokat_warehouses')
warehouses_latest = (
    warehouses
    .withColumn('rn', F.row_number().over(warehouses_window))
    .filter(F.col('rn') == 1)
    .drop('rn')
    .drop('file_date')
)

warehouse_bridge_base_pd = read_excel_standardized(
    path=BSNS_MAPPING_WH,
    sheet_name="точка- город",
    **WAREHOUSE_BRIDGE_CONFIG
)
warehouse_bridge_base_pd = normalize_string_id_column(warehouse_bridge_base_pd, "ds_customer_id")
warehouse_bridge_base_pd = normalize_string_id_column(warehouse_bridge_base_pd, "dc_customer_id")
if "ds_customer_id" in warehouse_bridge_base_pd.columns:
    warehouse_bridge_base_pd = warehouse_bridge_base_pd.query("ds_customer_id != 'Check'")
warehouse_bridge_base_pd = (
    warehouse_bridge_base_pd
    .dropna(subset=["warehouse_guid"])
    .drop_duplicates(subset=["warehouse_guid"])
)

dc_mapping_exceptions_pd = read_excel_standardized(
    path=BSNS_MAPPING_WH,
    sheet_name="исключения",
    **DC_EXCEPTIONS_CONFIG
)
dc_mapping_exceptions_pd = (
    dc_mapping_exceptions_pd
    .dropna(subset=["warehouse_guid"])
    .drop_duplicates(subset=["warehouse_guid"])
)

warehouse_bridge_pd = (
    warehouse_bridge_base_pd
    .merge(
        dc_mapping_exceptions_pd.rename(columns={"dc_warehouse_guid": "dc_warehouse_guid_override"}),
        on="warehouse_guid",
        how="left"
    )
    .assign(
        dc_warehouse_guid=lambda x: x["dc_warehouse_guid_override"].combine_first(x.get("dc_warehouse_guid")),
    )
    .drop(columns=["dc_warehouse_guid_override"], errors="ignore")
)

warehouse_bridge = spark.createDataFrame(warehouse_bridge_pd)

dc_to_ds_bridge = (
    warehouse_bridge
    .select("warehouse_guid", "dc_warehouse_guid", "dc_customer_id")
    .dropna(subset=["warehouse_guid"])
    .dropDuplicates(["warehouse_guid"])
)

In [0]:
products_latest.write.mode('overwrite').saveAsTable('ecom_etl.ds_samokat_products_latest')

In [0]:

warehouse_bridge_for_join = (
    warehouse_bridge
    .select(
        "warehouse_guid",
        F.col("ds_customer_id").alias("customer_id"),
        "dc_warehouse_guid",
        "dc_customer_id",
        F.col("gln_code").alias("bridge_gln_code"),
        "warehouse_format",
    )
    .dropDuplicates(["warehouse_guid"])
)

warehouses_latest = (
    warehouses_latest
    .join(warehouse_bridge_for_join, "warehouse_guid", "left")
    .withColumn("store_gln_code", F.coalesce(F.col("store_gln_code"), F.col("bridge_gln_code")))
    .drop("bridge_gln_code")
)


In [0]:
warehouses_latest.write.mode('overwrite').saveAsTable('ecom_etl.ds_samokat_warehouses_latest')

### FACT LAYER PREP


In [0]:

sales_stock_weekly = (
    spark.table('ecom_etl.ds_samokat_product_story')
    .filter(F.col('file_date') >= cutoff_source_date)
    .withColumn('osa_plan', F.lit(16))
    .withColumn(
        "start_of_week",
        F.date_add(F.date_trunc('week', F.col('file_date')), -7).cast('timestamp')
    )
    .groupBy(['start_of_week', 'warehouse_guid', 'product_guid'])
    .agg(
        F.sum('remnants').alias('remnants_ds'),
        F.mean('availability').alias('availability'),
        F.first('osa_plan').alias('osa_plan'),
        F.sum('sales_quantity').alias('sales_quantity'),
        F.sum('write_off_quantity').alias('write_off_quantity'),
        F.max('novelty_flg').alias('novelty_flg')
    )
)

csl_weekly_raw = (
    spark.table('ecom_etl.csl_delivery')
    .filter(F.col('required_date') >= cutoff_source_date)
    .where(F.lower('client_hierarchy_7') == "самокат")
    .withColumnsRenamed({
        'ean_piece_key': 'gtin',
        'customer_key': 'customer_id',
        'ordered_pieces': 'ordered',
        'delivered_pieces': 'delivered'
    })
    .withColumn('gtin', F.col('gtin').cast('string'))
    .withColumn('customer_id', F.col('customer_id').cast('string'))
    .withColumn('start_of_week', F.date_trunc('week', F.col("required_date")))
    .groupBy(['start_of_week', 'gtin', 'customer_id'])
    .agg(
        F.sum('ordered').cast('decimal(18,6)').alias('ordered'),
        F.sum('delivered').cast('decimal(18,6)').alias('delivered')
    )
)

dc_remnants_weekly = (
    spark.table('ecom_etl.ds_samokat_dc_remnants')
    .filter(F.col('file_date') >= cutoff_source_date)
    .withColumnRenamed('dc_warehouse_id', 'dc_warehouse_guid')
    .withColumn(
        "start_of_week",
        F.date_trunc('week', F.col('file_date')).cast('timestamp')
    )
    .groupBy(['start_of_week', 'dc_warehouse_guid', 'product_guid'])
    .agg(F.sum('quantity').alias('remnants_dc'))
)

products_by_gtin = (
    products_latest
    .select('product_guid', 'gtin')
    .dropna(subset=['gtin'])
    .dropDuplicates(['gtin', 'product_guid'])
)


In [0]:

csl_direct_ds = (
    csl_weekly_raw.alias('csl')
    .join(
        warehouse_bridge.select('warehouse_guid', 'ds_customer_id').dropna(subset=['ds_customer_id']).dropDuplicates(),
        F.col('csl.customer_id') == F.col('ds_customer_id'),
        'inner'
    )
    .join(products_by_gtin, 'gtin', 'left')
    .groupBy('start_of_week', 'warehouse_guid', 'product_guid')
    .agg(
        F.sum('ordered').cast('decimal(18,6)').alias('ordered_ds_direct'),
        F.sum('delivered').cast('decimal(18,6)').alias('delivered_ds_direct')
    )
)

csl_dc_raw = (
    csl_weekly_raw.alias('csl')
    .join(
        warehouse_bridge.select('dc_warehouse_guid', 'dc_customer_id').dropna(subset=['dc_customer_id']).dropDuplicates(),
        F.col('csl.customer_id') == F.col('dc_customer_id'),
        'inner'
    )
    .join(products_by_gtin, 'gtin', 'left')
    .groupBy('start_of_week', 'dc_warehouse_guid', 'product_guid')
    .agg(
        F.sum('ordered').cast('decimal(18,6)').alias('ordered_dc_raw'),
        F.sum('delivered').cast('decimal(18,6)').alias('delivered_dc_raw')
    )
)

dc_allocation_window = Window.partitionBy('start_of_week', 'dc_warehouse_guid', 'product_guid')

dc_allocation_base = (
    csl_dc_raw
    .select('start_of_week', 'dc_warehouse_guid', 'product_guid')
    .distinct()
    .join(
        dc_to_ds_bridge.select('warehouse_guid', 'dc_warehouse_guid').dropna(subset=['dc_warehouse_guid']).dropDuplicates(),
        'dc_warehouse_guid',
        'inner'
    )
    .join(
        sales_stock_weekly.select('start_of_week', 'warehouse_guid', 'product_guid', 'sales_quantity'),
        ['start_of_week', 'warehouse_guid', 'product_guid'],
        'left'
    )
    .withColumn('sales_base', F.coalesce(F.col('sales_quantity'), F.lit(0.0)))
    .withColumn('total_sales_dc', F.sum('sales_base').over(dc_allocation_window))
    .withColumn('ds_count', F.size(F.collect_set('warehouse_guid').over(dc_allocation_window)))
    .withColumn(
        'allocation_share',
        F.when(F.col('total_sales_dc') > 0, F.col('sales_base') / F.col('total_sales_dc'))
        .otherwise(F.lit(1.0) / F.col('ds_count'))
    )
    .select('start_of_week', 'dc_warehouse_guid', 'warehouse_guid', 'product_guid', 'allocation_share')
)

csl_dc_allocated = (
    csl_dc_raw
    .join(dc_allocation_base, ['start_of_week', 'dc_warehouse_guid', 'product_guid'], 'inner')
    .groupBy('start_of_week', 'warehouse_guid', 'product_guid')
    .agg(
        F.sum(F.col('ordered_dc_raw') * F.col('allocation_share')).cast('decimal(18,6)').alias('ordered_dc_alloc'),
        F.sum(F.col('delivered_dc_raw') * F.col('allocation_share')).cast('decimal(18,6)').alias('delivered_dc_alloc')
    )
)


In [0]:

dc_remnants_window = Window.partitionBy('start_of_week', 'dc_warehouse_guid', 'product_guid')

dc_remnants_allocated = (
    dc_remnants_weekly
    .join(
        dc_to_ds_bridge.select('warehouse_guid', 'dc_warehouse_guid').dropna(subset=['dc_warehouse_guid']).dropDuplicates(),
        'dc_warehouse_guid',
        'inner'
    )
    .withColumn('ds_count', F.size(F.collect_set('warehouse_guid').over(dc_remnants_window)))
    .withColumn('remnants_dc_per_ds_lag', F.try_divide(F.col('remnants_dc'), F.col('ds_count')))
    .groupBy('start_of_week', 'warehouse_guid', 'product_guid')
    .agg(F.sum('remnants_dc_per_ds_lag').alias('remnants_dc_per_ds_lag'))
)


In [ ]:

JOIN_COLS = ['product_guid', 'warehouse_guid', 'start_of_week']

remnants_ds_natural = (
    sales_stock_weekly
    .filter(F.col('remnants_ds').isNotNull())
    .select(
        'product_guid',
        'warehouse_guid',
        F.date_add('start_of_week', 7).alias('start_of_week'),
        F.col('remnants_ds').alias('remnants_ds_lag')
    )
)

fact_keys = (
    sales_stock_weekly.select(*JOIN_COLS)
    .unionByName(csl_direct_ds.select(*JOIN_COLS))
    .unionByName(csl_dc_allocated.select(*JOIN_COLS))
    .unionByName(dc_remnants_allocated.select(*JOIN_COLS))
    .distinct()
)

facts_base = (
    fact_keys
    .join(sales_stock_weekly, JOIN_COLS, 'left')
    .join(remnants_ds_natural, JOIN_COLS, 'left')
    .join(dc_remnants_allocated, JOIN_COLS, 'left')
    .join(csl_direct_ds, JOIN_COLS, 'left')
    .join(csl_dc_allocated, JOIN_COLS, 'left')
    .withColumn('ordered_ds_direct', F.coalesce(F.col('ordered_ds_direct'), F.lit(0)).cast('decimal(18,6)'))
    .withColumn('delivered_ds_direct', F.coalesce(F.col('delivered_ds_direct'), F.lit(0)).cast('decimal(18,6)'))
    .withColumn('ordered_dc_alloc', F.coalesce(F.col('ordered_dc_alloc'), F.lit(0)).cast('decimal(18,6)'))
    .withColumn('delivered_dc_alloc', F.coalesce(F.col('delivered_dc_alloc'), F.lit(0)).cast('decimal(18,6)'))
    .withColumn('ordered', (F.col('ordered_ds_direct') + F.col('ordered_dc_alloc')).cast('decimal(18,6)'))
    .withColumn('delivered', (F.col('delivered_ds_direct') + F.col('delivered_dc_alloc')).cast('decimal(18,6)'))
)

## ADD MATRIX

In [0]:
matrix_df = (
    spark.table('ecom_etl.ds_samokat_matrix')
)
max_date = matrix_df.select(F.max('file_date')).collect()[0][0]

In [0]:
matrix_df = matrix_df.filter(F.col('file_date') == max_date)
matrix_df = matrix_df.withColumn('is_actual_matrix', F.lit(1))
matrix_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('ecom_etl.ds_samokat_matrix_actual')

## FORECAST AND FINAL DATASET


In [0]:

dfm = (
    spark.table('ecom_etl.ds_samokat_dfm')
    .filter(F.col('file_date') >= cutoff_source_date)
    .withColumn(
        'forecast_lag',
        F.when(F.col('forecast_lag') == -48, 4).otherwise(F.col('forecast_lag'))
    )
    .withColumn('forecast_ml', F.coalesce(F.col('forecast_total'), F.col('forecast')))
    .withColumn(
        'forecast_cpfr',
        F.when(
            (F.col('forecast_regular').isNotNull()) & (F.col('forecast_regular') > F.col('forecast_ml')),
            F.col('forecast_regular')
        ).otherwise(F.col('forecast_ml'))
    )
)

dfm_pivot = (
    dfm
    .withColumn(
        "start_of_week",
        F.expr("date_add(date_trunc('week', file_date), CAST((forecast_lag - 1) * 7 AS INT))").cast('timestamp')
    )
    .groupBy('product_guid', 'warehouse_guid', 'start_of_week')
    .pivot('forecast_lag', [2, 3, 4])
    .agg(F.last('forecast_ml').alias('forecast_ml'), F.last('forecast_cpfr').alias('forecast_cpfr'))
    .select(
        'product_guid',
        'warehouse_guid',
        'start_of_week',
        '2_forecast_ml',
        '3_forecast_ml',
        '4_forecast_ml',
        '2_forecast_cpfr'
    )
)


In [0]:

base_mart = (
    dfm_pivot
    .join(facts_base, JOIN_COLS, how='full')
    .join(products_latest, ['product_guid'], how='left')
    .join(warehouses_latest, ['warehouse_guid'], how='left')
)


In [0]:

ds_per_dc_window = Window.partitionBy('product_guid', 'dc_warehouse_guid', 'start_of_week')
so_window = Window.partitionBy("product_guid", "warehouse_guid").orderBy("start_of_week").rowsBetween(-3, -1)

joined = (
    base_mart
    .withColumn(
        'osa_fact',
        F.when(
            F.col('availability').isNotNull() & F.col('osa_plan').isNotNull(),
            F.col('availability') * F.col('osa_plan')
        )
    )
    .withColumn(
        'osa_loss',
        F.when(F.col('osa_plan').isNotNull(), F.col('osa_plan') - F.col('osa_fact'))
    )
    .withColumn(
        "ds_per_dc",
        F.when(
            F.col("dc_warehouse_guid").isNotNull(),
            F.size(F.collect_set('warehouse_guid').over(ds_per_dc_window))
        )
    )
    .withColumn(
        'remnants_total',
        F.coalesce(F.col('remnants_ds_lag'), F.lit(0)) + F.coalesce(F.col('remnants_dc_per_ds_lag'), F.lit(0))
    )
    .withColumn('sell_out_average', F.avg(F.col('sales_quantity')).over(so_window))
    .withColumn("date_week", F.date_format("start_of_week", "YYYY-ww"))
    .withColumn(
        'ml_fa_numerator_lag_2',
        F.when(
            F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('2_forecast_ml'), F.lit(0)))
        )
    )
    .withColumn(
        'ml_fa_numerator_lag_3',
        F.when(
            F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('3_forecast_ml'), F.lit(0)))
        )
    )
    .withColumn(
        'ml_fa_numerator_lag_4',
        F.when(
            F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('4_forecast_ml'), F.lit(0)))
        )
    )
    .withColumn(
        'cpfr_fa_numerator_lag_2',
        F.when(
            F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('2_forecast_cpfr'), F.lit(0)))
        )
    )
)


## ADD PRICES

In [0]:
darkstore_city_to_state = {
    "Аксай": "Ростов-на-Дону",
    "Альметьевск": "Альметьевск",
    "Анапа": "Анапа",
    "Астрахань": "Астрахань",
    "Балаково": "Балаково",
    "Балашиха": "Москва",
    "Барнаул": "Барнаул",
    "Батайск": "Ростов-на-Дону",
    "Белгород": "Белгород",
    "Бердск": "Новосибирск",
    "Берёзовский муниципальный округ": "Екатеринбург",
    "Брянск": "Брянск",
    "Великий Новгород": "Великий Новгород",
    "Видное": "Москва",
    "Владимир": "Владимир",
    "Волгоград": "Волгоград",
    "Волжский": "Волгоград",
    "Волжский район": "Самара",
    "Вологда": "Вологда",
    "Воронеж": "Воронеж",
    "Всеволожский район": "Санкт-Петербург",
    "Гатчина": "Санкт-Петербург",
    "Геленджик": "Геленджик",
    "Губкинский городской округ": "Старый Оскол",
    "Дзержинск": "Нижний Новгород",
    "Дзержинский": "Москва",
    "Долгопрудный": "Москва",
    "Домодедово": "Москва",
    "Егорьевск": "Москва",
    "Екатеринбург": "Екатеринбург",
    "Жуковский": "Москва",
    "Зеленоград": "Москва",
    "Зеленодольск": "Казань",
    "Зеленодольский район": "Казань",
    "Иваново": "Иваново",
    "Ивантеевка": "Москва",
    "Истра": "Москва",
    "Казань": "Казань",
    "Калуга": "Калуга",
    "Кемерово": "Кемерово",
    "Киров": "Киров",
    "Киришский район": "Санкт-Петербург",
    "Кольцово": "Новосибирск",
    "Коломна": "Коломна",
    "Колпино": "Санкт-Петербург",
    "Королёв": "Москва",
    "Копейск": "Челябинск",
    "Кострома": "Кострома",
    "Котельники": "Москва",
    "Красногорск": "Москва",
    "Краснодар": "Краснодар",
    "Красноярск": "Красноярск",
    "Кронштадт": "Санкт-Петербург",
    "Кстово": "Нижний Новгород",
    "Кудрово": "Санкт-Петербург",
    "Курган": "Курган",
    "Курск": "Курск",
    "Ленинский городской округ": "Москва",
    "Липецк": "Липецк",
    "Лобня": "Москва",
    "Ломоносов": "Санкт-Петербург",
    "Лыткарино": "Москва",
    "Люберцы": "Москва",
    "Магнитогорск": "Магнитогорск",
    "Медведевский район": "Йошкар-Ола",
    "Москва": "Москва",
    "Московская область": "Москва",
    "Московский": "Москва",
    "Мурино": "Санкт-Петербург",
    "Мытищи": "Москва",
    "Набережные Челны": "Набережные Челны",
    "Наро-Фоминский городской округ": "Москва",
    "Нефтекамск": "Нефтекамск",
    "Нефтеюганск": "Нефтеюганск",
    "Нижневартовск": "Нижневартовск",
    "Нижнекамск": "Нижнекамск",
    "Нижний Новгород": "Нижний Новгород",
    "Нижний Тагил": "Нижний Тагил",
    "Новгородская область": "Великий Новгород",
    "Новокузнецк": "Новокузнецк",
    "Новокуйбышевск": "Самара",
    "Новосибирск": "Новосибирск",
    "Новоалтайск": "Барнаул",
    "Новоусманский район": "Воронеж",
    "Новороссийск": "Новороссийск",
    "Новочебоксарск": "Новочебоксарск",
    "Новая Адыгея": "Краснодар",
    "Ногинск": "Ногинск",
    "Обнинск": "Обнинск",
    "Одинцово": "Москва",
    "Одинцовский городской округ": "Москва",
    "Омск": "Омск",
    "Орел": "Орел",
    "Оренбург": "Оренбург",
    "Орехово-Зуево": "Орехово-Зуево",
    "Пенза": "Пенза",
    "Пензенский район": "Пенза",
    "Пермский муниципальный округ": "Пермь",
    "Пермь": "Пермь",
    "Первоуральск": "Екатеринбург",
    "Петрозаводск": "Петрозаводск",
    "Подольск": "Москва",
    "поселок Бугрино": "Санкт-Петербург",
    "поселок Бугры": "Санкт-Петербург",
    "поселок Верхнетемерницкий": "Ростов-на-Дону",
    "поселок Зональная Станция": "Томск",
    "поселок Зареченский": "Тамбов",
    "поселок Красный Бор": "Санкт-Петербург",
    "поселок Тельмана": "Санкт-Петербург",
    "пгт Яблоновский": "Краснодар",
    "Псков": "Псков",
    "Пушкино": "Москва",
    "Пушкин": "Санкт-Петербург",
    "Раменское": "Москва",
    "Реутов": "Москва",
    "Ростов-на-Дону": "Ростов-на-Дону",
    "Рязань": "Рязань",
    "Салават": "Салават",
    "Самара": "Самара",
    "Санкт-Петербург": "Санкт-Петербург",
    "Саранск": "Саранск",
    "Саратов": "Саратов",
    "Семилукский район": "Воронеж",
    "Сертолово": "Санкт-Петербург",
    "Сестрорецк": "Санкт-Петербург",
    "Сириус": "Сочи",
    "Смоленск": "Смоленск",
    "Солнечногорск": "Москва",
    "Сосновский район": "Челябинск",
    "Сочи": "Сочи",
    "Ставрополь": "Ставрополь",
    "Старый Оскол": "Старый Оскол",
    "Строитель": "Белгород",
    "Стерлитамак": "Стерлитамак",
    "Сургут": "Сургут",
    "Таганрог": "Таганрог",
    "Тахтамукайский район": "Краснодар",
    "Тамбов": "Тамбов",
    "Татарстан": "Казань",
    "Тверь": "Тверь",
    "Тольятти": "Тольятти",
    "Томск": "Томск",
    "Томский район": "Томск",
    "Троицк": "Москва",
    "Тула": "Тула",
    "Тюмень": "Тюмень",
    "Тюменский район": "Тюмень",
    "Ульяновск": "Ульяновск",
    "Уфа": "Уфа",
    "Ханты-Мансийск": "Ханты-Мансийск",
    "Химки": "Москва",
    "Чебоксары": "Чебоксары",
    "Челябинск": "Челябинск",
    "Челябинская область": "Челябинск",
    "Череповец": "Череповец",
    "Щелково": "Москва",
    "Щёлково": "Москва",
    "Щербинка": "Москва",
    "Электросталь": "Москва",
    "Энгельс": "Энгельс",
    "Ярославль": "Ярославль",
    "гп Новоселье": "Санкт-Петербург",
    "городской округ Азов": "Ростов-на-Дону",
    "городской округ Балашиха": "Москва",
    "городской округ Бердск": "Новосибирск",
    "городской округ Верхняя Пышма": "Екатеринбург",
    "городской округ Дзержинский": "Москва",
    "городской округ Домодедово": "Москва",
    "городской округ Екатеринбург": "Екатеринбург",
    "городской округ Ижевск": "Ижевск",
    "городской округ Истра": "Москва",
    "городской округ Котельники": "Москва",
    "городской округ Кохма": "Иваново",
    "городской округ Курск": "Курск",
    "городской округ Люберцы": "Москва",
    "городской округ Мегион": "Нижневартовск",
    "городской округ Мытищи": "Москва",
    "городской округ Новоалтайск": "Барнаул",
    "городской округ Обь": "Новосибирск",
    "городской округ Орёл": "Орел",
    "городской округ Солнечногорск": "Москва",
    "городской округ Фрязино": "Москва",
    "городской округ Жигулёвск": "Тольятти",
    "деревня Анкудиновка": "Нижний Новгород",
    "деревня Афонино": "Нижний Новгород",
    "деревня Новое Девяткино": "Санкт-Петербург",
    "деревня Патрушева": "Тюмень",
    "деревня Татаренкова": "Курск",
    "село Андреевка": "Москва",
    "село Засечное": "Пенза",
}

In [0]:
mapping_df = spark.createDataFrame(
    [(k, v) for k, v in darkstore_city_to_state.items()],
    ["darkstore_city", "city_nm"]
)

In [0]:
start_date = joined.select(F.min('start_of_week')).first()[0]

In [0]:
brand = spark.table('ecom_etl.def_brand').filter(F.col('holding')=='PepsiCo')
product = spark.table('ecom_etl.def_product').join(F.broadcast(brand), on='brand_id', how='inner').select('product_id', 'barcode', 'product_name', 'brand_name')

darkstore = spark.table('ecom_etl.def_darkstore').filter(F.col('darkstore_group_platform')=='Самокат')
darkstore_mapped = (
    darkstore
    .join(mapping_df, on="darkstore_city", how="left")
)
shelf = (
    spark.table('ecom_etl.def_e_shelf')
    .filter(F.col('date') >= start_date)
    .join(product, on="product_id", how='inner')
    .join(F.broadcast(darkstore_mapped), on='darkstore_id', how='inner')
    .select(
        'product_id',
        "barcode", 
        "product_name", 
        'brand_name', 
        'date', 
        'customer_id',
        'city_nm', 
        'price_without_promo',
        'promo_price', 
        'discount_percent'
    )
)

weekly_shelf = (
    shelf
    .filter(F.dayofweek(F.col("date")) > 4)
    .withColumn('start_of_week', F.date_trunc('week', F.col("date")))
    .groupBy(
        "start_of_week",
        "barcode",
        "city_nm",
    )
    .agg(
        F.mode("price_without_promo").cast('double').alias("price_without_promo"),
        F.mode("promo_price").cast('double').alias("promo_price"),
        F.mode("discount_percent").cast('double').alias("discount_percent"),
    )
    .withColumnRenamed('barcode', 'gtin')
)

In [0]:
joined_with_price = (
    joined
    .join(weekly_shelf, on=["start_of_week", "gtin", 'city_nm'], how='left')
    .withColumn("_thursday", F.date_add("start_of_week", 3))
    .withColumn("date_year", F.year("_thursday"))
    .withColumn("date_month", F.month("_thursday"))
    .drop("_thursday")
)

# METRICS

In [0]:

full_df_city = (
    joined_with_price
    .groupBy(
        'date_week',
        'date_year',
        'date_month',
        'start_of_week',
        'city_nm',
        'gtin',
        'product_guid'
    )
    .agg(
        *[
            F.sum(col).alias(col)
            for col in [
                "osa_plan",
                "osa_fact",
                "osa_loss",
                "ordered_ds_direct",
                "delivered_ds_direct",
                "ordered_dc_alloc",
                "delivered_dc_alloc",
                "ordered",
                "delivered",
                "sales_quantity",
                "sell_out_average",
                "remnants_ds",
                "remnants_ds_lag",
                "remnants_dc_per_ds_lag",
                "remnants_total",
                '2_forecast_ml',
                '3_forecast_ml',
                '4_forecast_ml',
                '2_forecast_cpfr',
                'ml_fa_numerator_lag_2',
                'ml_fa_numerator_lag_3',
                'ml_fa_numerator_lag_4',
                'cpfr_fa_numerator_lag_2',
            ]
        ],
        F.mean("promo_price").alias('mean_promo_price'),
        F.mean("price_without_promo").alias("mean_price_without_promo"),
        F.mean("discount_percent").alias("mean_discount_percent"),
        F.avg('sales_quantity').alias('avg_sell_out_per_warehouse'),
        F.count(F.when(F.col('sales_quantity').isNotNull(), F.col('warehouse_guid'))).alias('warehouse_count'),
        F.sum(
            F.when(
                (F.col('sell_out_average') > 0) &
                ((F.col('remnants_total') / F.col('sell_out_average')) * 7 >= 3),
                1
            ).otherwise(0)
        ).alias('isa_fact')
    )
)


In [0]:

final_metrics_df = (
    joined_with_price
    .withColumn('shortfalls_total', F.col('ordered') - F.col('delivered'))
    .withColumn('shortfalls_direct', F.col('ordered_ds_direct') - F.col('delivered_ds_direct'))
    .withColumn('shortfalls_dc', F.col('ordered_dc_alloc') - F.col('delivered_dc_alloc'))
    .withColumn('shortfalls_so', F.col('ordered') - F.col('delivered'))
    .withColumn('shortfalls_ds', F.col('ordered_ds_direct') - F.col('delivered_ds_direct'))
)

full_metrics_city_df = (
    full_df_city
    .withColumn('shortfalls_total', F.col('ordered') - F.col('delivered'))
    .withColumn('shortfalls_direct', F.col('ordered_ds_direct') - F.col('delivered_ds_direct'))
    .withColumn('shortfalls_dc', F.col('ordered_dc_alloc') - F.col('delivered_dc_alloc'))
    .withColumn('shortfalls_so', F.col('ordered') - F.col('delivered'))
    .withColumn('shortfalls_ds', F.col('ordered_ds_direct') - F.col('delivered_ds_direct'))
)


## VOLUME MAPPING

In [0]:

cols = [
    "sales_quantity",
    "remnants_ds_lag",
    "remnants_dc_per_ds_lag",
    "remnants_total",
    "ordered_ds_direct",
    "delivered_ds_direct",
    "ordered_dc_alloc",
    "delivered_dc_alloc",
    "ordered",
    "delivered",
    "2_forecast_ml",
    "3_forecast_ml",
    "4_forecast_ml",
]

final_metrics_df_mapped = apply_volume_mapping(
    final_metrics_df,
    sap_vol_mapping,
    cols
)

full_metrics_city_df_mapped = apply_volume_mapping(
    full_metrics_city_df,
    sap_vol_mapping,
    cols
)


# WRITE DATA

## COLS TO WRITE

### COMMON COLS

In [0]:

date_cols = [
    "date_week",
    "date_year",
    "date_month",
    "start_of_week",
]

service_cols_common = [
    "osa_plan",
    "osa_fact",
    "osa_loss",
    "ordered_ds_direct",
    "delivered_ds_direct",
    "ordered_dc_alloc",
    "delivered_dc_alloc",
    "ordered",
    "delivered",
    "shortfalls_total",
    "shortfalls_direct",
    "shortfalls_dc",
    "shortfalls_so",
    "shortfalls_ds",
]

sales_stock_cols = [
    "sales_quantity",
    "sell_out_average",
    "remnants_ds_lag",
    "remnants_dc_per_ds_lag",
    "remnants_total",
]

forecast_cols = [
    "2_forecast_ml",
    "3_forecast_ml",
    "4_forecast_ml",
    "2_forecast_cpfr",
    "ml_fa_numerator_lag_2",
    "ml_fa_numerator_lag_3",
    "ml_fa_numerator_lag_4",
    "cpfr_fa_numerator_lag_2",
]

vol_metric_cols = [
    "sales_quantity_vol",
    "remnants_ds_lag_vol",
    "remnants_dc_per_ds_lag_vol",
    "remnants_total_vol",
    "ordered_ds_direct_vol",
    "delivered_ds_direct_vol",
    "ordered_dc_alloc_vol",
    "delivered_dc_alloc_vol",
    "ordered_vol",
    "delivered_vol",
    "2_forecast_ml_vol",
    "3_forecast_ml_vol",
    "4_forecast_ml_vol",
]


### WH_COLS

In [0]:

final_id_cols = [
    "product_guid",
    "gtin",
    "sap_id",
    "warehouse_guid",
    "customer_id",
    "dc_warehouse_guid",
    "dc_customer_id",
    "warehouse_format",
    "city_nm",
]

final_price_cols = [
    "price_without_promo",
    "promo_price",
    "discount_percent",
]

final_select_cols = (
    date_cols
    + final_id_cols
    + final_price_cols
    + service_cols_common
    + sales_stock_cols
    + forecast_cols
    + vol_metric_cols
)


### CITY_COLS

In [0]:

city_id_cols = [
    "product_guid",
    "gtin",
    "sap_id",
    "city_nm",
    "warehouse_count",
]

city_price_cols = [
    "mean_price_without_promo",
    "mean_promo_price",
    "mean_discount_percent",
]

city_service_cols = [
    "osa_plan",
    "osa_fact",
    "osa_loss",
    "isa_fact",
    "ordered_ds_direct",
    "delivered_ds_direct",
    "ordered_dc_alloc",
    "delivered_dc_alloc",
    "ordered",
    "delivered",
    "shortfalls_total",
    "shortfalls_direct",
    "shortfalls_dc",
    "shortfalls_so",
    "shortfalls_ds",
]

city_select_cols = (
    date_cols
    + city_id_cols
    + city_price_cols
    + city_service_cols
    + sales_stock_cols
    + forecast_cols
    + vol_metric_cols
)


## FULL METRICS

In [0]:
spark.sql(f"DELETE FROM {TRGT_TABLE_METRICS} WHERE date_week >= '{cutoff_output}'")


In [0]:
(
    final_metrics_df_mapped
    .filter(F.col("start_of_week") >= cutoff_source_date)
    .select(*final_select_cols)
    .filter(F.col("product_guid").isNotNull())
    .filter(F.col("warehouse_guid").isNotNull())
    .dropDuplicates(["date_week", "product_guid", "warehouse_guid"])
    .write.mode(write_mode)
    .option("mergeSchema", "true")
    .saveAsTable(TRGT_TABLE_METRICS)
)

## CITY METRICS

In [0]:
spark.sql(f"DELETE FROM {TRGT_TABLE_METRCICS_CITY} WHERE date_week >= '{cutoff_output}'")


In [0]:

(
    full_metrics_city_df_mapped
    .filter(F.col("date_week") >= cutoff_output)
    .select(*city_select_cols)
    .filter(F.col("gtin").isNotNull())
    .write.mode(write_mode)
    .option("mergeSchema", "true")
    .saveAsTable(TRGT_TABLE_METRCICS_CITY)
)

# ALERTS

## DOS-ALERTS

In [0]:

last_5_weeks = (
    final_metrics_df
    .dropna(subset=['remnants_ds_lag'])
    .select('date_week')
    .distinct()
    .orderBy(F.col('date_week').desc())
    .limit(5)
    .collect()
)
last_5_weeks = [row['date_week'] for row in last_5_weeks]
last_2_weeks = last_5_weeks[:2]

base_df = (
    final_metrics_df
    .filter(F.col('date_week').isin(last_5_weeks))
    .withColumn('osa', F.try_divide(F.col('osa_fact'), F.col('osa_plan')))
    .withColumn('sl', F.try_divide(F.col('delivered'), F.col('ordered')))
    .withColumn('dos', 7 * F.try_divide(F.col('remnants_total'), F.col('sell_out_average')))
    .withColumn(
        'is_alert',
        (F.col('osa') < 0.95) & ((F.col('sl') > 0.8) | F.col('sl').isNull()) & (F.col('dos') < 6)
    )
)

w0 = last_2_weeks[0]
w1 = last_2_weeks[1]

alert_pairs = (
    base_df
    .filter(F.col('date_week').isin(last_2_weeks))
    .groupBy('warehouse_guid', 'product_guid')
    .agg(
        F.max(F.when((F.col('date_week') == w0) & F.col('is_alert'), 1).otherwise(0)).alias('alert_w0'),
        F.max(F.when((F.col('date_week') == w1) & F.col('is_alert'), 1).otherwise(0)).alias('alert_w1'),
    )
    .filter((F.col('alert_w0') == 1) | (F.col('alert_w1') == 1))
    .select('warehouse_guid', 'product_guid')
)

alerting_df = (
    base_df
    .join(alert_pairs, ['warehouse_guid', 'product_guid'], 'inner')
    .select(
        'date_week', 'warehouse_guid', 'product_guid', "osa_plan", "osa_fact", "ordered",
        'osa', 'sl', 'dos', 'sales_quantity', 'remnants_total'
    )
    .orderBy('warehouse_guid', 'product_guid', 'date_week')
)


In [0]:
alerting_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TRGT_TABLE_ALERTS)

## FORECAST CHECK

In [0]:

full_df = (
    final_metrics_df
    .join(
        spark.table('ecom_etl.td_product').select('gtin', 'category').distinct(),
        on='gtin',
        how='inner'
    )
)

sell_out_df = full_df.dropna(subset=['sales_quantity'])

sell_out_weeks = [
    row['date_week'] for row in
    sell_out_df.select('date_week').distinct().orderBy(F.desc('date_week')).collect()
]

cutoff_rank  = sell_out_weeks[5]
cutoff_select = sell_out_weeks[9]

top_sellers = (
    sell_out_df
    .filter(F.col('date_week') >= F.lit(cutoff_rank))
    .groupBy('category', 'gtin')
    .agg(F.sum('sales_quantity').alias('total_sales'))
    .withColumn('rnk', F.row_number().over(
        Window.partitionBy('category').orderBy(F.desc('total_sales'))
    ))
    .filter(F.col('rnk') <= 8)
    .select('gtin')
)

result_df = (
    full_df
    .filter(F.col('date_week') >= F.lit(cutoff_select))
    .join(top_sellers, on=['gtin'], how='inner')
    .withColumn('osa', F.try_divide(F.col('osa_fact'), F.col('osa_plan')))
    .withColumn('sl', F.try_divide(F.col('delivered'), F.col('ordered')))
    .withColumn('dos', 7 * F.try_divide(F.col('remnants_total'), F.col('sell_out_average')))
    .select(
        'date_week', 'warehouse_guid', 'product_guid', 'gtin', "osa_plan", "osa_fact", "ordered", "3_forecast_ml", "2_forecast_cpfr",
        'osa', 'sl', 'dos', 'sales_quantity', 'remnants_total'
    )
    .orderBy('warehouse_guid', 'gtin', 'date_week')
)


In [0]:
result_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TRGT_TABLE_FORECAST_CHECK)

In [0]:
dbutils.notebook.exit('success')

# TEST ZONE

## CSL CHECK

In [0]:

orphan_rows = joined.filter(
    F.col('warehouse_guid').isNull() & F.col('ordered').isNotNull()
)


In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
display(orphan_rows)


In [0]:

orphan_rows = joined.filter(
    F.col('warehouse_guid').isNull() & F.col('ordered').isNotNull()
)
orphan_customer_ids = orphan_rows.select('customer_id').distinct()

orphan_stats = orphan_rows.agg(
    F.count("*").alias("orphan_count"),
    F.sum("ordered").alias("orphan_ordered"),
    F.sum("delivered").alias("orphan_delivered")
).first()

orphan_count = orphan_stats["orphan_count"]
orphan_ordered = orphan_stats["orphan_ordered"] or 0
orphan_delivered = orphan_stats["orphan_delivered"] or 0

raw_csl_totals = (
    csl_weekly_raw
    .agg(
        F.sum("ordered").alias("total_ordered"),
        F.sum("delivered").alias("total_delivered")
    )
    .first()
)

total_ordered_raw = raw_csl_totals["total_ordered"] or 0
loss_pct = 100 * orphan_ordered / (total_ordered_raw or 1)

summary_df = spark.createDataFrame(
    [
        ("Orphan CSL-строк", orphan_count),
        ("Заказов в orphan", f"{orphan_ordered:,.0f}"),
        ("Поставок в orphan", f"{orphan_delivered:,.0f}"),
        ("Всего заказов в CSL", f"{total_ordered_raw:,.0f}"),
        ("Потери заказов", f"{loss_pct:.1f}%")
    ],
    ["Metric", "Value"]
)
display(summary_df)

display(
    orphan_rows
    .groupBy("customer_id")
    .agg(
        F.sum("ordered").alias("ordered"),
        F.sum("delivered").alias("delivered"),
        F.countDistinct("gtin").alias("gtins"),
        F.countDistinct("start_of_week").alias("weeks")
    )
    .orderBy(F.desc("ordered"))
    .limit(20)
)


In [0]:

display(
    orphan_customer_ids.alias("o")
    .join(
        warehouse_bridge.select(F.col("ds_customer_id").alias("customer_id")).withColumn("matched_as_ds", F.lit(True)),
        "customer_id",
        "left"
    )
    .join(
        warehouse_bridge.select(F.col("dc_customer_id").alias("customer_id")).withColumn("matched_as_dc", F.lit(True)),
        "customer_id",
        "left"
    )
    .withColumn("matched_as_ds", F.col("matched_as_ds").isNotNull())
    .withColumn("matched_as_dc", F.col("matched_as_dc").isNotNull())
    .groupBy("matched_as_ds", "matched_as_dc")
    .count()
)

display(
    spark.table('ecom_etl.csl_delivery')
    .withColumn('customer_id', F.col('customer_key').cast('string'))
    .join(orphan_customer_ids, on='customer_id', how='inner')
    .select('customer_id', 'city', 'address')
    .dropDuplicates(['customer_id'])
    .orderBy('city', 'customer_id')
)


In [0]:

sample_id = '200892226'

fresh_bridge_pd = read_excel_standardized(
    path=BSNS_MAPPING_WH,
    sheet_name="точка- город",
    **WAREHOUSE_BRIDGE_CONFIG
)
fresh_bridge_pd = normalize_string_id_column(fresh_bridge_pd, "ds_customer_id")
fresh_bridge_pd = normalize_string_id_column(fresh_bridge_pd, "dc_customer_id")

print(fresh_bridge_pd[(fresh_bridge_pd['ds_customer_id'] == sample_id) | (fresh_bridge_pd['dc_customer_id'] == sample_id)])
print(f"\nВсего строк в файле сейчас: {len(fresh_bridge_pd)}")
print(f"Строк в warehouse_bridge Spark: {warehouse_bridge.count()}")

display(warehouses_latest.filter(F.col('customer_id') == sample_id))

wh_guid = warehouses_latest.filter(F.col('customer_id') == sample_id).select('warehouse_guid').first()
if wh_guid:
    wguid = wh_guid['warehouse_guid']
    print(f"warehouse_guid: {wguid}")
    print(f"Строк в facts_base: {facts_base.filter(F.col('warehouse_guid') == wguid).count()}")
    print(f"Строк в dfm_pivot:  {dfm_pivot.filter(F.col('warehouse_guid') == wguid).count()}")
else:
    print("warehouse_guid не найден в warehouses_latest")


In [0]:

fresh_bridge_spark = spark.createDataFrame(fresh_bridge_pd)

display(
    orphan_customer_ids
    .join(
        fresh_bridge_spark.select(F.col('ds_customer_id').alias('customer_id')).withColumn('in_fresh_ds_file', F.lit(True)),
        'customer_id',
        'left'
    )
    .join(
        fresh_bridge_spark.select(F.col('dc_customer_id').alias('customer_id')).withColumn('in_fresh_dc_file', F.lit(True)),
        'customer_id',
        'left'
    )
    .withColumn('in_fresh_ds_file', F.col('in_fresh_ds_file').isNotNull())
    .withColumn('in_fresh_dc_file', F.col('in_fresh_dc_file').isNotNull())
    .groupBy('in_fresh_ds_file', 'in_fresh_dc_file')
    .count()
)


In [0]:

raw_csl_totals = (
    csl_weekly_raw
    .groupBy("start_of_week")
    .agg(
        F.sum("ordered").alias("raw_ordered"),
        F.sum("delivered").alias("raw_delivered")
    )
)

joined_totals = (
    joined
    .groupBy("start_of_week")
    .agg(
        F.sum("ordered").alias("joined_ordered"),
        F.sum("delivered").alias("joined_delivered"),
        F.sum("ordered_ds_direct").alias("joined_ordered_ds_direct"),
        F.sum("ordered_dc_alloc").alias("joined_ordered_dc_alloc")
    )
)

display(
    raw_csl_totals
    .join(joined_totals, on="start_of_week", how="left")
    .withColumn(
        "coverage_total_pct",
        100 * F.try_divide(F.col("joined_ordered"), F.col("raw_ordered"))
    )
    .orderBy(F.desc("start_of_week"))
    .limit(12)
)


## FA_CHECK

## FA RAW VALIDATION

In [ ]:

# ── 1. Сырые продажи (повторяем логику cell-25) ──
raw_sales = (
    spark.table('ecom_etl.ds_samokat_product_story')
    .filter(F.col('file_date') >= cutoff_source_date)
    .withColumn(
        "start_of_week",
        F.date_add(F.date_trunc('week', F.col('file_date')), -7).cast('timestamp')
    )
    .groupBy('start_of_week', 'warehouse_guid', 'product_guid')
    .agg(F.sum('sales_quantity').alias('sales_quantity'))
)

# ── 2. Сырые прогнозы (повторяем логику cell-34) ──
raw_dfm = (
    spark.table('ecom_etl.ds_samokat_dfm')
    .filter(F.col('file_date') >= cutoff_source_date)
    .withColumn(
        'forecast_lag',
        F.when(F.col('forecast_lag') == -48, 4).otherwise(F.col('forecast_lag'))
    )
    .withColumn('forecast_ml', F.coalesce(F.col('forecast_total'), F.col('forecast')))
    .withColumn(
        'forecast_cpfr',
        F.when(
            (F.col('forecast_regular').isNotNull()) & (F.col('forecast_regular') > F.col('forecast_ml')),
            F.col('forecast_regular')
        ).otherwise(F.col('forecast_ml'))
    )
    .withColumn(
        "start_of_week",
        F.expr("date_add(date_trunc('week', file_date), CAST((forecast_lag - 1) * 7 AS INT))").cast('timestamp')
    )
    .groupBy('product_guid', 'warehouse_guid', 'start_of_week')
    .pivot('forecast_lag', [2, 3, 4])
    .agg(F.last('forecast_ml').alias('forecast_ml'), F.last('forecast_cpfr').alias('forecast_cpfr'))
    .select(
        'product_guid', 'warehouse_guid', 'start_of_week',
        '2_forecast_ml', '3_forecast_ml', '4_forecast_ml', '2_forecast_cpfr'
    )
)

# ── 3. FULL JOIN сырых данных + FA-числители ──
raw_joined = (
    raw_sales
    .join(raw_dfm, ['product_guid', 'warehouse_guid', 'start_of_week'], 'full')
    .withColumn("_thursday", F.date_add("start_of_week", 3))
    .withColumn("date_year", F.year("_thursday"))
    .withColumn("date_month", F.month("_thursday"))
    .drop("_thursday")
    .withColumn('ml_fa_num_lag_2',
        F.when(F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('2_forecast_ml'), F.lit(0)))))
    .withColumn('ml_fa_num_lag_3',
        F.when(F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('3_forecast_ml'), F.lit(0)))))
    .withColumn('ml_fa_num_lag_4',
        F.when(F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('4_forecast_ml'), F.lit(0)))))
    .withColumn('cpfr_fa_num_lag_2',
        F.when(F.col('sales_quantity').isNotNull(),
            F.abs(F.col('sales_quantity') - F.coalesce(F.col('2_forecast_cpfr'), F.lit(0)))))
)

# ── 4. Присоединяем products_latest → product_df для бренда ──
raw_with_brand = (
    raw_joined
    .join(products_latest, 'product_guid', 'left')
    .join(product_df, 'gtin', 'left')
)

# ── 5. Агрегация до brand × month и расчёт FA ──
fa_check_raw = (
    raw_with_brand
    .groupBy('date_year', 'date_month', 'brand_description')
    .agg(
        F.sum('sales_quantity').alias('sales_quantity'),
        F.sum('2_forecast_ml').alias('2_forecast_ml'),
        F.sum('3_forecast_ml').alias('3_forecast_ml'),
        F.sum('4_forecast_ml').alias('4_forecast_ml'),
        F.sum('2_forecast_cpfr').alias('2_forecast_cpfr'),
        F.sum('ml_fa_num_lag_2').alias('ml_fa_numerator_lag_2'),
        F.sum('ml_fa_num_lag_3').alias('ml_fa_numerator_lag_3'),
        F.sum('ml_fa_num_lag_4').alias('ml_fa_numerator_lag_4'),
        F.sum('cpfr_fa_num_lag_2').alias('cpfr_fa_numerator_lag_2'),
    )
    .withColumn('fa_lag2', 1 - F.try_divide(F.col('ml_fa_numerator_lag_2'), F.col('sales_quantity')))
    .withColumn('fa_lag3', 1 - F.try_divide(F.col('ml_fa_numerator_lag_3'), F.col('sales_quantity')))
    .withColumn('fa_lag4', 1 - F.try_divide(F.col('ml_fa_numerator_lag_4'), F.col('sales_quantity')))
    .withColumn('fa_cpfr_lag2', 1 - F.try_divide(F.col('cpfr_fa_numerator_lag_2'), F.col('sales_quantity')))
    .orderBy('date_year', 'date_month', 'brand_description')
)

display(fa_check_raw)

In [ ]:

# ── FA CHECK: pipeline (fa_check) vs raw (fa_check_raw) ──
# Проверка: если fa_check уже определён выше, сравниваем с raw
# Если нет — просто показываем raw результат

try:
    fa_check  # проверяем что переменная существует
    
    comparison = (
        fa_check.alias('pipe')
        .join(
            fa_check_raw.alias('raw'),
            ['date_year', 'date_month', 'brand_description'],
            'full'
        )
        .select(
            'date_year', 'date_month', 'brand_description',
            F.col('pipe.sales_quantity').alias('pipe_sales'),
            F.col('raw.sales_quantity').alias('raw_sales'),
            F.round(F.col('pipe.sales_quantity') - F.col('raw.sales_quantity'), 2).alias('sales_diff'),
            F.round(F.col('pipe.fa_lag2'), 4).alias('pipe_fa2'),
            F.round(F.col('raw.fa_lag2'), 4).alias('raw_fa2'),
            F.round(F.col('pipe.fa_lag2') - F.col('raw.fa_lag2'), 4).alias('fa2_diff'),
            F.round(F.col('pipe.fa_lag3'), 4).alias('pipe_fa3'),
            F.round(F.col('raw.fa_lag3'), 4).alias('raw_fa3'),
            F.round(F.col('pipe.fa_lag3') - F.col('raw.fa_lag3'), 4).alias('fa3_diff'),
            F.round(F.col('pipe.fa_lag4'), 4).alias('pipe_fa4'),
            F.round(F.col('raw.fa_lag4'), 4).alias('raw_fa4'),
            F.round(F.col('pipe.fa_lag4') - F.col('raw.fa_lag4'), 4).alias('fa4_diff'),
            F.round(F.col('pipe.fa_cpfr_lag2'), 4).alias('pipe_cpfr'),
            F.round(F.col('raw.fa_cpfr_lag2'), 4).alias('raw_cpfr'),
            F.round(F.col('pipe.fa_cpfr_lag2') - F.col('raw.fa_cpfr_lag2'), 4).alias('cpfr_diff'),
        )
        .orderBy('date_year', 'date_month', 'brand_description')
    )
    
    print("=== Строки с расхождением sales ===")
    display(comparison.filter(F.col('sales_diff') != 0))
    
    print("=== Строки с расхождением FA lag2 > 0.01 ===")
    display(comparison.filter(F.abs(F.col('fa2_diff')) > 0.01))

except NameError:
    print("fa_check не определён — показываем только raw результат")
    display(fa_check_raw)

In [ ]:

# ── 6. Сравнение: pipeline fa_check vs raw fa_check_raw ──
comparison = (
    fa_check.alias('pipe')
    .join(
        fa_check_raw.alias('raw'),
        ['date_year', 'date_month', 'brand_description'],
        'full'
    )
    .select(
        'date_year', 'date_month', 'brand_description',
        F.col('pipe.sales_quantity').alias('pipe_sales'),
        F.col('raw.sales_quantity').alias('raw_sales'),
        F.round(F.col('pipe.sales_quantity') - F.col('raw.sales_quantity'), 2).alias('sales_diff'),
        F.round(F.col('pipe.fa_lag2'), 4).alias('pipe_fa_lag2'),
        F.round(F.col('raw.fa_lag2'), 4).alias('raw_fa_lag2'),
        F.round(F.col('pipe.fa_lag2') - F.col('raw.fa_lag2'), 4).alias('fa2_diff'),
        F.round(F.col('pipe.fa_lag3'), 4).alias('pipe_fa_lag3'),
        F.round(F.col('raw.fa_lag3'), 4).alias('raw_fa_lag3'),
        F.round(F.col('pipe.fa_lag3') - F.col('raw.fa_lag3'), 4).alias('fa3_diff'),
        F.round(F.col('pipe.fa_lag4'), 4).alias('pipe_fa_lag4'),
        F.round(F.col('raw.fa_lag4'), 4).alias('raw_fa_lag4'),
        F.round(F.col('pipe.fa_lag4') - F.col('raw.fa_lag4'), 4).alias('fa4_diff'),
        F.round(F.col('pipe.fa_cpfr_lag2'), 4).alias('pipe_fa_cpfr'),
        F.round(F.col('raw.fa_cpfr_lag2'), 4).alias('raw_fa_cpfr'),
        F.round(F.col('pipe.fa_cpfr_lag2') - F.col('raw.fa_cpfr_lag2'), 4).alias('cpfr_diff'),
    )
    .orderBy('date_year', 'date_month', 'brand_description')
)

display(comparison.filter(F.col('sales_diff') != 0))

In [0]:
fa_check = (                                                                                                          
    joined_with_price                                                                                                 
    .join(product_df, on='gtin', how='left')                                                                          
    .groupBy('date_year', 'date_month', 'brand_name')                                                                 
    .agg(                                                                                                           
        F.sum('sales_quantity').alias('actual'),
        # NEW: числитель и знаменатель только по строкам где оба значения есть
        F.sum(F.when(                                                                                                 
            F.col('sales_quantity').isNotNull() & F.col('2_forecast_ml').isNotNull(),
            F.abs(F.col('sales_quantity') - F.col('2_forecast_ml'))                                                   
        )).alias('fa_num_lag2_new'),                                                                                  
        F.sum(F.when(
            F.col('2_forecast_ml').isNotNull(), F.col('sales_quantity')                                               
        )).alias('fa_denom_lag2_new'),                                                                              
        F.sum(F.when(                                                                                                 
            F.col('sales_quantity').isNotNull() & F.col('3_forecast_ml').isNotNull(),                                 
            F.abs(F.col('sales_quantity') - F.col('3_forecast_ml'))
        )).alias('fa_num_lag3_new'),                                                                                  
        F.sum(F.when(                                                                                                 
            F.col('3_forecast_ml').isNotNull(), F.col('sales_quantity')
        )).alias('fa_denom_lag3_new'),                                                                                
        # OLD: для сравнения — числитель с coalesce(0)                                                              
        F.sum(F.abs(                                                                                                  
            F.coalesce(F.col('sales_quantity'), F.lit(0)) - F.coalesce(F.col('2_forecast_ml'), F.lit(0))            
        )).alias('fa_num_lag2_old'),                                                                                  
        F.sum(F.coalesce(F.col('sales_quantity'), F.lit(0))).alias('fa_denom_old'),                                   
    )                                                                                                                 
    .withColumn('fa_lag2_new', 1 - F.try_divide(F.col('fa_num_lag2_new'), F.col('fa_denom_lag2_new')))                
    .withColumn('fa_lag3_new', 1 - F.try_divide(F.col('fa_num_lag3_new'), F.col('fa_denom_lag3_new')))              
    .withColumn('fa_lag2_old', 1 - F.try_divide(F.col('fa_num_lag2_old'), F.col('fa_denom_old')))                     
    .orderBy('date_year', 'date_month', 'brand_name')                                                               
)                                                                                                                     
                                                                                                                    
display(fa_check)                                 

In [0]:
# Aggregate phantom statistics
phantom_stats = (
    joined_with_price
    .agg(
        F.count("*").alias("total_rows"),
        F.count(
            F.when(
                F.col("sales_quantity").isNull() & F.col("2_forecast_ml").isNotNull(),
                1,
            )
        ).alias("forecast_no_actual"),
        F.count(
            F.when(
                F.col("sales_quantity").isNotNull() & F.col("2_forecast_ml").isNull(),
                1,
            )
        ).alias("actual_no_forecast"),
        F.count(
            F.when(
                F.col("sales_quantity").isNotNull() & F.col("2_forecast_ml").isNotNull(),
                1,
            )
        ).alias("both_present"),
    )
)
display(phantom_stats)

# Sum of phantom forecasts (lag2) and total actual sales
phantom_forecast_sum = (
    joined_with_price
    .filter(F.col("sales_quantity").isNull() & F.col("2_forecast_ml").isNotNull())
    .agg(fsum("2_forecast_ml").alias("s"))
    .first()["s"]
)

total_sales = (
    joined_with_price
    .agg(fsum("sales_quantity").alias("s"))
    .first()["s"]
)

# Output metrics
print(f"\nФантомный прогноз (lag2):   {float(phantom_forecast_sum or 0):,.0f}")
print(f"Реальные продажи:           {float(total_sales or 0):,.0f}")
print(f"Соотношение phantom/sales:  {float(phantom_forecast_sum or 0) / float(total_sales or 1):.2%}")
print("\nЕсли > 10% — старая мера FA была сильно искажена")

#AD_HOC

In [0]:
last_4_week = (
    spark.table('ecom_etl.ds_samokat_availability_metrics')
    .select('date_week')
    .distinct()
    .orderBy(F.desc('date_week'))
    .limit(4)
    .rdd.map(lambda r: r['date_week'])
    .collect()
)
display(
    spark.table('ecom_etl.ds_samokat_availability_metrics')
    .filter(F.col('date_week').isin(last_4_week))
    .filter(F.col('gtin')=='4600494694202'))

In [0]:
from pyspark.sql import functions as F

source = spark.table("ecom_etl.ds_samokat_availability_metrics")

last_4_weeks_df = (
    source
    .select("date_week")
    .where(
        F.col("date_week").isNotNull() & F.col('sales_quantity').isNotNull()
        )
    .distinct()
    .orderBy(F.desc("date_week"))
    .limit(4)
)

ad_hoc = (
    source
    .join(last_4_weeks_df, on="date_week", how="inner")
    .withColumn("dos", F.try_divide(F.col("remnants_total"), F.col("sell_out_average") * 7))
    .withColumn(
        "isa_bool",
        F.when(
            (F.col("sell_out_average") > 0) & (F.col("dos") >= 3),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .select(
        "start_of_week",
        "warehouse_guid",
        "city_nm",
        "product_guid",
        "gtin",
        "dos",
        "isa_bool",
        "osa_plan",
        "osa_fact",
        "sales_quantity",
        "remnants_total"
    )
)

In [0]:
display(ad_hoc)

In [0]:
ad_hoc.toPandas().to_csv('ad_hoc_2026_04_08.csv')


In [0]:
display(df_last_8)

In [0]:
spark.table('ecom_etl.ds_samokat_availability_metrics').filter(F.col('city_nm')=='Москва').filter(F.col('product_guid').isin([
    '0ffda4b2-6e59-11e9-80c5-0cc47a817925',
'd4ee76b9-2af9-11ee-b971-08c0eb32008b',
'f8efa3a1-ab72-11f0-a97c-be3af2b6059f'
])).filter(F.col('date_week') >='2025-48').filter(F.col('date_week') < '2025-52').toPandas().to_csv('ad_hoc_samokat.csv')

In [0]:

display(facts_base.filter(F.col('warehouse_guid') == '69b53b66-103f-11ee-b10a-08c0eb31fffb'))


In [0]:
display(warehouses_latest.filter(F.col('customer_id')==200892226))

In [0]:

display(joined.filter(F.col('customer_id') == '200892226').select('start_of_week').distinct())


In [0]:

display(
    csl_weekly_raw
    .filter(F.col('customer_id') == '200892226')
    .select('start_of_week')
    .distinct()
)


In [0]:

display(
    warehouse_bridge
    .filter((F.col('ds_customer_id') == '200892226') | (F.col('dc_customer_id') == '200892226'))
)


In [0]:

display(
    joined
    .filter(F.col('customer_id') == '200892226')
)


In [0]:
display(
    spark.table('ecom_etl.ds_samokat_warehouses').filter(F.col('store_gln_code')=='4610249279844')
)

In [0]:
display(
    spark.table('ecom_etl.csl_delivery')
    .filter(F.col('ean_piece_key').isin([4600494681172,4600494681226]))
    .filter(F.col('delivery_date_week') == '10/2026')
    .groupBy('ean_piece_key', 'delivery_date_week')
    .agg(F.sum('ordered_pieces').alias('ordered'), F.sum('delivered_pieces').alias('delivered'))
    .withColumn('sl', 100 * F.try_divide(F.col('delivered'), F.col('ordered')))
)

In [0]:
display(
    spark.table('ecom_etl.ds_samokat_availability_metrics_city')
    .filter(F.col('sap_id').isNull()).groupBy('gtin').agg(F.min('start_of_week'), F.max('start_of_week'))
)

In [0]:
display(
    dc_remnants
    .filter(F.col("product_guid")=='0ffda4b2-6e59-11e9-80c5-0cc47a817925')
    .groupBy("WeekNumber", 'product_guid')
    .agg(
        F.sum('quantity')
    )
)

In [0]:
lost_dc = [
    'abbb9e3e-bc1f-11ed-885d-08c0eb32014b', '7307ea02-496c-11ed-885a-08c0eb32014b', '0087831f-9906-11ee-8861-08c0eb32014b'
]

In [0]:
display(
    spark.table('ecom_etl.ds_samokat_availability_metrics')
    .filter(F.col('date_week') == '2026-10')
    .filter(F.col('product_guid') == '0ffda4b2-6e59-11e9-80c5-0cc47a817925')
    .groupBy('date_week', 'product_guid')
    .agg(F.sum('remnants_dc_per_ds_lag').alias('dc_remnants'), F.sum('remnants_total').alias('total_remnants'), F.sum('ordered'))
)